# lora-eval-lab: the GPU steps
---

The only two steps that need a GPU, in one notebook: base generation, training, tuned generation.

No logic lives here. Every stage cell calls the package, so the code under test is the code in the repo.

**Order matters** and mirrors `PROCESS.md`:
- The untuned base model generates first. That is the control.
- Training produces the adapter.
- The tuned model generates with the identical prompt and decoding.

## Stages and outputs
---

| Stage | Cell | Writes | Copied to Drive |
| --- | --- | --- | --- |
| Data | 3 | `data/raw/*.csv` (verified against pinned checksums) | no |
| Base generations | 4 | `results/generations_base.jsonl` | yes |
| Train | 5 | `adapter/`, `results/train_config.json`, `results/train_log.jsonl` | yes |
| Tuned generations | 6 | `results/generations_tuned.jsonl` | yes |

Runtime: T4 GPU (Runtime > Change runtime type > T4 GPU). Every stage copies its output to Drive as soon as it finishes, so a pre-emption costs one stage.

## 1. Setup
---

Installs Unsloth (which brings `trl`, `peft` and `bitsandbytes`), clones the repo, installs the package, mounts Drive.

When the Drive popup appears: choose the account, tick **Select all**, Continue. The mount fails if any box is left unticked.

In [ ]:
!pip install -q unsloth
!git clone -q https://github.com/J-Jurza/lora-eval-lab.git
%cd lora-eval-lab
!pip install -q -e .

import os
import shutil

from google.colab import drive, files
from IPython.display import Markdown, display

# A failed earlier mount leaves a stub directory that blocks the next attempt
if os.path.isdir("/content/drive") and not os.path.ismount("/content/drive"):
    shutil.rmtree("/content/drive")
drive.mount("/content/drive", force_remount=True)

## Settings you can change
---

| Setting | What it controls |
| --- | --- |
| `MODEL` | The base model. Changing it changes the experiment, and `DECISIONS.md` |
| `BATCH_SIZE` | Prompts per generate call. Lower it if the T4 runs out of memory |
| `DRIVE_DIR` | Where artefacts that must outlive the session are copied |

In [ ]:
MODEL      = "Qwen/Qwen2.5-1.5B-Instruct"
BATCH_SIZE = 8
DRIVE_DIR  = "/content/drive/MyDrive/lora-eval-lab"

!mkdir -p $DRIVE_DIR/results $DRIVE_DIR/adapter
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

display(Markdown(
    "**Active settings.**\n\n"
    f"- Model: **{MODEL}**\n"
    f"- Batch size: **{BATCH_SIZE}**\n"
    f"- Drive: `{DRIVE_DIR}`"
))

## 3. Data
---

Downloads the pinned CSVs and verifies their checksums. The held-out ids are already frozen in the repo, so nothing here can change which dialogues are the exam.

In [ ]:
!python -m lora_eval_lab.data --download --stats

## 4. Base generations (the control)
---

Untuned base model over the 199 held-out dialogues, greedy decoding. Resumable: rerun the cell if the session drops.

In [ ]:
!python -m lora_eval_lab.generate --tag base --model $MODEL --batch-size $BATCH_SIZE
!cp results/generations_base.jsonl $DRIVE_DIR/results/

## 5. Train
---

QLoRA on the 1,201 training rows, validation loss every 25 steps, adapter saved. Hyperparameters live in `train.CONFIG` and are echoed to `results/train_config.json`. Expect 15 to 30 minutes on a T4.

In [ ]:
!python -m lora_eval_lab.train --model $MODEL --out adapter
!cp -r adapter $DRIVE_DIR/ && cp results/train_config.json results/train_log.jsonl $DRIVE_DIR/results/

## 6. Tuned generations
---

Same prompt, same decoding, base model plus the adapter. Resumable like the base run.

In [ ]:
!python -m lora_eval_lab.generate --tag tuned --model $MODEL --adapter adapter --batch-size $BATCH_SIZE
!cp results/generations_tuned.jsonl $DRIVE_DIR/results/

## 7. Take the results home
---

Download the generations and commit them locally. Judging and metrics run on CPU.

In [ ]:
files.download("results/generations_base.jsonl")
files.download("results/generations_tuned.jsonl")
files.download("results/train_config.json")
files.download("results/train_log.jsonl")